In [ ]:
#Real-time processing code, that will load the trained parameters then make prediction on real-time data and send the reponse to emergency contacts if crash is detected over consecutive windows

import numpy as np
import tensorflow as tf
from collections import deque
import serial

model = tf.keras.models.load_model("helmX_autoencoder.keras")

mean = np.load("processed/mean.npy")
std  = np.load("processed/std.npy")
threshold = np.load("processed/threshold.npy")

In [ ]:
WINDOW_SIZE = 60
STRIDE = 15
CONSECUTIVE_REQUIRED = 3
ALERT_SENT = False

buffer = deque(maxlen=WINDOW_SIZE)
new_samples = 0
anomaly_count = 0

from mpu6050 import mpu6050

sensor = mpu6050(0x68)

In [ ]:
def read_sensor():
    accel = sensor.get_accel_data()
    gyro  = sensor.get_gyro_data()

    return [
        accel['x'], accel['y'], accel['z'],
        gyro['x'],  gyro['y'],  gyro['z']
    ]

In [ ]:
gsm = serial.Serial("/dev/ttyUSB0", baudrate=9600, timeout=1)

def send_sms(lat, lon):
    message = f"🚨 CRASH DETECTED!\nLocation:\nhttps://maps.google.com/?q={lat},{lon}"

    gsm.write(b'AT+CMGF=1\r')
    time.sleep(1)
    gsm.write(b'AT+CMGS="+923XXXXXXXXX"\r')
    time.sleep(1)
    gsm.write(message.encode())
    gsm.write(bytes([26]))

In [ ]:
def get_gps_coordinates():
    while True:
        line = gsm.readline().decode(errors='ignore')
        if "$GPGGA" in line:
            parts = line.split(",")
            if parts[2] and parts[4]:
                lat = float(parts[2][:2]) + float(parts[2][2:]) / 60
                lon = float(parts[4][:3]) + float(parts[4][3:]) / 60
                return lat, lon

In [ ]:
import time

while True:
    sample = read_sensor()
    buffer.append(sample)
    new_samples += 1

    if len(buffer) < WINDOW_SIZE:
        time.sleep(0.01)
        continue

    if new_samples < STRIDE:
        time.sleep(0.01)
        continue

    new_samples = 0

    window = np.array(buffer)
    window = (window - mean) / std
    window = np.transpose(window, (1,0))
    window = window[np.newaxis, ..., np.newaxis]

    recon = model.predict(window, verbose=0)
    mse = np.mean((window - recon)**2)

    if mse > threshold:
        anomaly_count += 1
    else:
        anomaly_count = 0
        ALERT_SENT = False

    if anomaly_count >= CONSECUTIVE_REQUIRED and not ALERT_SENT:
        print("🚨 CRASH DETECTED 🚨")
        anomaly_count = 0

        lat, lon = get_gps_coordinates()
        print("📍 Location:", lat, lon)

        send_sms(lat, lon)

        ALERT_SENT = True

        break

    time.sleep(0.01)